# Design visualizations

This Jupyter notebook is for designing visualizations for the streamlit app only. Final visualization code is copied into app.py.

In [ ]:
# Forecast chart using Altair

import pandas as pd
import altair as alt
import datetime as dt
import sys
import os

# Import the predict module 
cwd = os.getcwd()
if cwd.endswith('webapp_streamlit'):
    os.chdir('..')
from Predict.predict import make_predictions

# Load the predictions
# predictions_df = pd.read_csv('predictions.csv')
# predictions_df['date'] = pd.to_datetime(predictions_df['date'])
# # Filter to only show today onwards
# today = pd.Timestamp(dt.date.today())
# # filtered_df = predictions_df[predictions_df['date'] >= today]
# filtered_df = predictions_df
# # TESTING
# # filtered_df = predictions_df[(predictions_df['date'] >= pd.Timestamp(dt.date(2025, 12, 24))) & (predictions_df['date'] <= pd.Timestamp(dt.date(2025, 12, 24) + dt.timedelta(days=14)))]

# Predict
print("Calling prediction pipeline...")
predictions_df = make_predictions(dt.date.today(), dt.date.today() + dt.timedelta(days=14))
predictions_df.to_csv('webapp_streamlit/predictions.csv', index=False)

# Create a base layer with the shared X-axis (formatted as Day-Date, no vertical grid)
base = alt.Chart(predictions_df).encode(
    x=alt.X('date:T', axis=alt.Axis(format='%a %d', title=None, grid=False, labelAngle=-45, tickCount=14))
)

# Define bands for background annotation (for colored zones only, no text labels on bands)
bands = [
    {"y0": 0, "y1": 10, "color": "#CFD6EA"},
    {"y0": 10, "y1": 40, "color": "#EDF7ED"},
    {"y0": 40, "y1": 70, "color": "#D3EBCD"},
    {"y0": 70, "y1": 100, "color": "#AED5A2"},
]
band_chart = alt.Chart(pd.DataFrame(bands)).mark_rect().encode(
    y='y0:Q',
    y2='y1:Q',
    color=alt.Color('color:N', scale=None, legend=None),
).properties(
    width=alt.Step(30),
    height=150
)

# Add thin dark gray horizontal lines at each y tick
yticks = [0, 10, 40, 70, 100]
ytick_labels = {
    0: "LOW",
    10: "ELEVATED",
    40: "HIGH",
    70: "VERY HIGH",
    100: ""
}
lines_df = pd.DataFrame({'y': yticks})
lines_chart = alt.Chart(lines_df).mark_rule(strokeWidth=1, color='#444', opacity=0.65).encode(
    y='y:Q'
).properties(
    width=alt.Step(30),
    height=150
)

porcini_line = base.mark_line(
    color='#5C3A21', strokeWidth=3, interpolate='monotone'
).encode(
    y=alt.Y(
        'porcini_index:Q',
        title='Fruiting Likelihood',
        scale=alt.Scale(domain=[0, 100]),
        axis=alt.Axis(
            grid=False,
            values=yticks,
            tickCount=len(yticks),
            ticks=True,
            labels=True,
            labelExpr=f"""
                {{
                    0: '{ytick_labels[0]}',
                    10: '{ytick_labels[10]}',
                    40: '{ytick_labels[40]}',
                    70: '{ytick_labels[70]}',
                    100: '{ytick_labels[100]}'
                }}[datum.value] || ''
            """,
            labelBaseline="bottom",   # Move the labels up above the ticks
            labelPadding=0            # Reduce padding (default is 4), move closer to axis (and up)
        )
    )
).properties(height=150)

porcini_points = base.mark_point(
    color='#5C3A21', filled=True, size=60
).encode(
    y=alt.Y('porcini_index:Q')
).properties(height=150)

porcini = band_chart + lines_chart + porcini_line + porcini_points

# Create a DataFrame for grid lines at 0, 5, 10, 15, 20, 25
temp_grid_y = [0, 5, 10, 15, 20, 25]
temp_grid_df = pd.DataFrame({'y': temp_grid_y})
temp_grid_chart = alt.Chart(temp_grid_df).mark_rule(
    strokeWidth=1,
    color='#333',
    opacity=0.45
).encode(
    y='y:Q'
).properties(
    height=100
)

temp = (
    temp_grid_chart
    + base.mark_line(
        color='#FF4B4B', strokeWidth=1.5, interpolate='monotone'
    ).encode(
        y=alt.Y(
            'tmax_c_true:Q',
            title='Temp (°C)',
            scale=alt.Scale(domain=[0, 25]),
            axis=alt.Axis(grid=False)
        )
    ).properties(height=100)
    + base.mark_point(
        color='#FF4B4B', filled=True, size=30
    ).encode(
        y=alt.Y('tmax_c_true:Q', scale=alt.Scale(domain=[0, 25]))
    ).properties(height=100)
    + base.mark_line(
        color='#0077B6', strokeWidth=1.5, interpolate='monotone'
    ).encode(
        y=alt.Y(
            'tmin_c_true:Q',
            title='Temp (°C)',
            scale=alt.Scale(domain=[0, 25]),
            axis=alt.Axis(grid=False)
        )
    ).properties(height=100)
    + base.mark_point(
        color='#0077B6', filled=True, size=30
    ).encode(
        y=alt.Y('tmin_c_true:Q', scale=alt.Scale(domain=[0, 25]))
    ).properties(height=100)
)

# Set rain y-axis domain: [0, max(10, prcp_mm_true.max())]
rain_y_max = max(10, predictions_df['prcp_mm_true'].max())
rain = base.mark_bar(
    color='#3A86FF', opacity=0.8
).encode(
    y=alt.Y(
        'prcp_mm_true:Q',
        title='Rain (mm)',
        scale=alt.Scale(domain=[0, rain_y_max]),
        axis=alt.Axis(grid=False)
    )
).properties(height=100)

# If no rain is forecasted at all, overlay gray text
if predictions_df['prcp_mm_true'].max() == 0:
    no_rain_text = alt.Chart(pd.DataFrame({'x': [predictions_df['date'].iloc[len(predictions_df)//2]], 'y': [rain_y_max/2]})).mark_text(
        text="No rain in forecast",
        color='gray',
        size=18,
        fontWeight='bold'
    ).encode(
        x='x:T',
        y='y:Q'
    ).properties(height=100)
    rain = rain + no_rain_text

# Stack them vertically, share the X-axis, and remove outer borders
forecast_chart = alt.vconcat(
    porcini, temp, rain, spacing=10
).resolve_scale(
    x='shared' # This forces them to align perfectly
).configure_view(
    strokeOpacity=0 # Removes the box around the charts
)

# st.altair_chart(forecast_chart, use_container_width=True)

display(forecast_chart)

In [ ]:
# Save the chart as a JSON file
forecast_chart.save('webapp_streamlit/visualizations/forecast_chart.json')